# Gemini API

## Import API Key

In [13]:
import os
from dotenv import load_dotenv

# Load any local environment variables
load_dotenv(".env")

# Get the API key from the environment variables
api_key = os.getenv("GEMINI_API_KEY")

## Analyse Coin Function

In [14]:
import time
from google import genai
from PIL import Image

def analyse_coin(image1_path, image2_path):

    prompt = """
    The two images show the obverse and reverse sides of a coin. Please can you analyse the images and populate a JSON object with the following values:
    
    - country: The coins country of origin (string)
    - year: The year the coin was minted (integer)
    - denomination: The name given to a coin of this value (string)
    - materials: A list of up to 3 metals found within the coin, each represented as an object with a material (string) and a percentage (float)
    - estimated_value: The estimated value of the coin in GBP (float)
    - rarity: On a scale from 1 (common) to 10 (extremely rare), how rare is this coin (integer)
    - obverse_image: A very short description of the image on the obverse side of the coin (string) - nullable
    - obverse_text: A transcript of the text on the obverse side of the coin (string) - nullable
    - reverse_image: A very short description of the image on the reverse side of the coin (string) - nullable
    - reverse_text: A transcript of the text on the reverse side of the coin (string) - nullable
    - confidence: On a scale from 0 (not confident) to 100 (certain) how confident are you in your analysis and valuation of this coin (integer)
    - context: A short paragraph explaining the reasoning behind why the coin has been identified and valued in this way. If applicable, mention further steps to prove or disprove the identification (string)
    
    If any of the nullable fields are not relevant, populate the value with null.
    """

    image1 = Image.open(image1_path)
    image2 = Image.open(image2_path)

    client = genai.Client(api_key=api_key)

    # Try 3 times, each time it fails (due to the api being unavailable), wait exponentially longer
    for attempt in range(25):
        try:
            return client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[prompt, image1, image2]
            )
        except Exception as e:
            if "503" in str(e):
                time.sleep(2 ** attempt)
            else:
                raise
    
    return None

## Coin 1 Test

The images show a 1965 Elizabeth II Crown with Winston Churchill on the reverse side. These images were found on Ebay and the listing was £2.

<div style="display: flex; justify-content: left;">
    <img src="./images/coin1-obverse.jpg" style="width: 30%; object-fit: cover;">
    <img src="./images/coin1-reverse.jpg" style="width: 30%; object-fit: cover;">
</div>

In [15]:
response = analyse_coin(
    'images/coin1-obverse.jpg',
    'images/coin1-reverse.jpg'
)

if response:
    print(response.text)

```json
{
  "country": "United Kingdom",
  "year": 1965,
  "denomination": "Crown",
  "materials": [
    {
      "material": "Copper",
      "percentage": 75.0
    },
    {
      "material": "Nickel",
      "percentage": 25.0
    }
  ],
  "estimated_value": 1.00,
  "rarity": 1,
  "obverse_image": "A right-facing laureate bust of Queen Elizabeth II.",
  "obverse_text": "ELIZABETH II DEI GRATIA REGINA F.D. 1965",
  "reverse_image": "A right-facing bust of Winston Churchill.",
  "reverse_text": "CHURCHILL",
  "confidence": 100,
  "context": "This coin is a 1965 British Crown, issued to commemorate the death of Sir Winston Churchill. The obverse features the second portrait of Queen Elizabeth II by Arnold Machin, while the reverse shows a portrait of Winston Churchill. It is made of cupro-nickel. Due to its very high mintage (over 19 million for circulation, plus significant numbers of uncirculated and proof versions), this coin is extremely common and holds very little numismatic value in

## Coin 2 Test

The images show a Henry VIII Hammered Groat Facing Portrait Tower Coin. These images were found on Ebay and the listing was £10.

<div style="display: flex; justify-content: left;">
    <img src="./images/coin2-obverse.jpg" style="width: 30%; object-fit: cover;">
    <img src="./images/coin2-reverse.jpg" style="width: 30%; object-fit: cover;">
</div>

In [16]:
response = analyse_coin(
    'images/coin2-obverse.jpg',
    'images/coin2-reverse.jpg'
)

if response:
    print(response.text)

```json
{
  "country": "England",
  "year": 1545,
  "denomination": "Groat",
  "materials": [
    {
      "material": "Silver",
      "percentage": 33.3
    },
    {
      "material": "Copper",
      "percentage": 66.7
    }
  ],
  "estimated_value": 300.0,
  "rarity": 6,
  "obverse_image": "Crowned, bearded bust of King Henry VIII facing forward.",
  "obverse_text": "HENRIC VIII DI GRA REX AGL FR Z HIB",
  "reverse_image": "Quartered shield of arms featuring the fleurs-de-lis of France and the lions of England, topped with a cross design.",
  "reverse_text": "POSUI DEUM ADJUTOREM MEUM",
  "confidence": 95,
  "context": "This coin is an English Groat from the Third Coinage of King Henry VIII, minted between 1544 and 1547. The obverse clearly depicts a crude, crowned, and bearded bust of Henry VIII, characteristic of this later coinage period, along with parts of the legend 'HENRIC VIII DI GRA REX AGL FR Z HIB' (Henry VIII by the Grace of God King of England, France, and Ireland). The r

## Coin 3 Test

The images show a 1917 King George V Half Penny Coin. These images were found on Ebay and the listing was £1.80.

<div style="display: flex; justify-content: left;">
    <img src="./images/coin3-obverse.jpg" style="width: 30%; object-fit: cover;">
    <img src="./images/coin3-reverse.jpg" style="width: 30%; object-fit: cover;">
</div>

In [17]:
response = analyse_coin(
    'images/coin3-obverse.jpg',
    'images/coin3-reverse.jpg'
)

if response:
    print(response.text)

```json
{
  "country": "United Kingdom",
  "year": 1917,
  "denomination": "Half Penny",
  "materials": [
    {
      "material": "Copper",
      "percentage": 95.5
    },
    {
      "material": "Tin",
      "percentage": 3.0
    },
    {
      "material": "Zinc",
      "percentage": 1.5
    }
  ],
  "estimated_value": 2.00,
  "rarity": 1,
  "obverse_image": "A left-facing bust of King George V.",
  "obverse_text": "GEORGIVS V DEI GRA: BRITT: OMN: REX FID: DEF: IND: IMP:",
  "reverse_image": "Britannia seated facing right, holding a trident and shield.",
  "reverse_text": "ONE HALF PENNY 1917",
  "confidence": 100,
  "context": "The coin is a British Half Penny from 1917. The obverse features King George V, identified by the inscription 'GEORGIVS V DEI GRA: BRITT: OMN: REX FID: DEF: IND: IMP:', which translates to 'George V, by the Grace of God, King of all the Britains, Defender of the Faith, Emperor of India'. The reverse shows Britannia, a common personification of Britain, seated 

## Coin 4 Test

The images show a Solid Silver 1oz Canadian Maple Leaf 2017 Five Dollars Bullion Coin. These images were found on Ebay and the listing was £84.

<div style="display: flex; justify-content: left;">
    <img src="./images/coin4-obverse.jpg" style="width: 30%; object-fit: cover;">
    <img src="./images/coin4-reverse.jpg" style="width: 30%; object-fit: cover;">
</div>

In [18]:
response = analyse_coin(
    'images/coin4-obverse.jpg',
    'images/coin4-reverse.jpg'
)

if response:
    print(response.text)

```json
{
  "country": "Canada",
  "year": 2017,
  "denomination": "5 Dollars",
  "materials": [
    {
      "material": "Silver",
      "percentage": 99.99
    }
  ],
  "estimated_value": 24.00,
  "rarity": 2,
  "obverse_image": "A right-facing profile portrait of Queen Elizabeth II.",
  "obverse_text": "ELIZABETH II 5 DOLLARS 2017",
  "reverse_image": "A large, detailed maple leaf.",
  "reverse_text": "CANADA 9999 FINE SILVER 1 OZ ARGENT PUR",
  "confidence": 95,
  "context": "This coin is identifiable as a Canadian Silver Maple Leaf due to the clear portrait of Queen Elizabeth II on the obverse, along with the denomination of '5 DOLLARS' and the year '2017'. The reverse unmistakably features a prominent maple leaf, the country name 'CANADA', and inscriptions '9999 FINE SILVER 1 OZ ARGENT PUR', which confirm it is a one-troy-ounce coin made of 99.99% pure silver. These coins are widely recognized bullion products, and their value is primarily driven by the fluctuating spot price of s

## Coin 5 Test

The images show a Claudius II, Roman imperial coin. These images were found on Ebay and the listing was £15.

<div style="display: flex; justify-content: left;">
    <img src="./images/coin5-obverse.jpg" style="width: 30%; object-fit: cover;">
    <img src="./images/coin5-reverse.jpg" style="width: 30%; object-fit: cover;">
</div>

In [19]:
response = analyse_coin(
    'images/coin5-obverse.jpg',
    'images/coin5-reverse.jpg'
)

if response:
    print(response.text)

```json
{
  "country": "Roman Empire",
  "year": 278,
  "denomination": "Antoninianus",
  "materials": [
    {
      "material": "Bronze",
      "percentage": 95.0
    },
    {
      "material": "Silver",
      "percentage": 5.0
    }
  ],
  "estimated_value": 45.0,
  "rarity": 3,
  "obverse_image": "Radiate bust of Emperor Probus facing right.",
  "obverse_text": "IMP C PROBVS P F AVG",
  "reverse_image": "Standing female figure, possibly Providentia or Pax, holding an object.",
  "reverse_text": "PROVIDENTIA AVG",
  "confidence": 85,
  "context": "The coin has been identified as a Roman Antoninianus of Emperor Probus, who reigned from 276 to 282 AD. The identification is primarily based on the obverse depiction of a radiate bust, characteristic of Antoniniani from the 3rd century, and the partially visible obverse legend 'IMP C PROBVS P F AVG', clearly showing 'PROBVS'. The coin's material is typical for a late 3rd-century Antoninianus, which was a billon coin, heavily debased with a